# Supervised Model Evaluation

## Final Narrative-Subset Model Evaluation


## Notebook Overview

This evaluation notebook is the narrative-subset companion to the all-records final evaluation notebook.

It evaluates the best narrative-only Random Forest on complaints where `narrative_present == 1`, which lets us ask whether lightweight text features add meaningful value once every record in scope actually contains narrative text.


### Evaluation Metrics

Because this is still a classification problem with two response outcomes, we need more than accuracy to understand model quality. This evaluation reports:

- `accuracy` for overall correctness
- `precision` for how often predicted untimely cases are truly untimely
- `recall` for how many true untimely cases are captured
- `f1` as the primary model-selection metric because it balances precision and recall
- `roc_auc` to summarize ranking quality across thresholds
- `average_precision` because precision-recall behavior is informative under heavy imbalance
- `balanced_accuracy` so minority-class performance is not washed out by the majority class

Per the project requirements, model-family comparison should rely on cross-validated summary statistics. In this notebook, the saved development results provide that comparison, while the deeper evaluation focuses only on the selected Random Forest.


In [ ]:
import ast
import os
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "consumer_banking_relief.parquet"
COMPARISON_PATH = PROJECT_ROOT / "output_tables" / "final_relief_narrative_supervised_model_comparison.csv"
MODEL_ARTIFACT_PATH = PROJECT_ROOT / "data" / "processed" / "best_final_relief_narrative_response_model.joblib"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(exist_ok=True)
OUTPUT_TABLES_DIR = PROJECT_ROOT / "output_tables"
OUTPUT_TABLES_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
CV_FOLDS = 5
FAST_CV_FOLDS = 3
MAX_MODEL_ROWS = 120000
N_JOBS = min(4, max(1, (os.cpu_count() or 2) - 1))

assert DATA_PATH.exists(), f"Expected source parquet at {DATA_PATH}"
raw_df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(raw_df):,} banking complaints from {DATA_PATH.name}")


### Model Comparison

The development notebook compared four model families using cross-validated metrics. We reload that saved comparison here so the overall-results section remains concise and consistent, then narrow the rest of the notebook to the selected Random Forest.


In [ ]:
if COMPARISON_PATH.exists():
    comparison_df = pd.read_csv(COMPARISON_PATH)
    display(comparison_df)
    rf_comparison_row = comparison_df.loc[comparison_df["model_family"].eq("Random Forest")].copy()
    comparison_df.round(4).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_model_family_comparison.csv", index=False)
else:
    comparison_df = None
    rf_comparison_row = pd.DataFrame()
    print("Saved comparison table not found. Run the development notebook first if you want to display the model-family comparison here.")

rf_comparison_row


Even before the deeper analysis, the development comparison matters for the report because it shows that the selected model was not chosen from intuition alone. In the final narrative-only workflow, the comparison is especially useful because it isolates the value of structured context plus lightweight narrative features on the subset where narratives are actually present.


### Best Model Deep Dive

To keep evaluation consistent with development, we rebuild the same feature set and train/test split used in the development notebook. We keep only simple narrative indicators here, since more advanced narrative feature generation is being handled separately.


In [ ]:
def make_one_hot_encoder():
    kwargs = {"handle_unknown": "ignore"}
    try:
        return OneHotEncoder(sparse_output=True, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=True, **kwargs)


POSITIVE_RELIEF_VALUES = ["Closed with monetary relief"]
TOPIC_CLUSTER_CANDIDATES = [
    "narrative_topic_cluster",
    "topic_cluster",
    "complaint_topic_cluster",
    "narrative_cluster",
]
POSITIVE_WORDS = {
    "help",
    "helped",
    "resolved",
    "resolution",
    "fixed",
    "refund",
    "reversed",
    "credit",
    "waived",
    "approved",
    "released",
    "corrected",
    "responded",
    "support",
}
NEGATIVE_WORDS = {
    "fraud",
    "error",
    "late",
    "denied",
    "foreclosure",
    "harassment",
    "problem",
    "issue",
    "fees",
    "charged",
    "wrong",
    "delay",
    "delayed",
    "unauthorized",
    "scam",
    "misleading",
    "debt",
    "threat",
}


def compute_sentiment_score(text_series):
    cleaned = (
        text_series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z\s]", " ", regex=True)
    )
    token_lists = cleaned.str.split()
    positive_hits = token_lists.apply(lambda words: sum(word in POSITIVE_WORDS for word in words))
    negative_hits = token_lists.apply(lambda words: sum(word in NEGATIVE_WORDS for word in words))
    token_count = token_lists.str.len().clip(lower=1)
    return ((positive_hits - negative_hits) / token_count).astype(float)


def prepare_model_df(df):
    model_df = df.copy()
    model_df["Date received"] = pd.to_datetime(model_df["Date received"], errors="coerce")
    model_df["target_relief"] = model_df["Company response to consumer"].isin(POSITIVE_RELIEF_VALUES).astype(int)

    text_series = model_df["Consumer complaint narrative"].fillna("")
    model_df["narrative_present"] = text_series.str.len().gt(0).astype(int)
    model_df["narrative_char_count"] = text_series.str.len()
    model_df["narrative_word_count"] = text_series.str.split().str.len().fillna(0)
    model_df["narrative_sentiment_score"] = compute_sentiment_score(text_series)

    topic_cluster_features = [col for col in TOPIC_CLUSTER_CANDIDATES if col in model_df.columns]
    for col in topic_cluster_features:
        model_df[col] = model_df[col].fillna("missing").astype(str)

    model_df["received_year"] = model_df["Date received"].dt.year
    model_df["received_month"] = model_df["Date received"].dt.month
    model_df["received_quarter"] = model_df["Date received"].dt.quarter
    model_df["received_dayofweek"] = model_df["Date received"].dt.dayofweek
    model_df["received_day"] = model_df["Date received"].dt.day

    top_category_limits = {
        "Company": 100,
        "Sub-product": 35,
        "Issue": 45,
        "Sub-issue": 70,
    }
    for col, top_n in top_category_limits.items():
        values = model_df[col].fillna("missing").astype(str)
        keep = set(values.value_counts().head(top_n).index)
        model_df[col] = values.where(values.isin(keep), other="__OTHER__")
    model_df["Submitted via"] = model_df["Submitted via"].fillna("missing").astype(str)
    model_df["Product"] = model_df["Product"].fillna("missing").astype(str)

    model_df = model_df.dropna(subset=["Date received", "target_relief"]).copy()
    if len(model_df) > MAX_MODEL_ROWS:
        _, model_df = train_test_split(
            model_df,
            test_size=MAX_MODEL_ROWS,
            stratify=model_df["target_relief"],
            random_state=RANDOM_STATE,
        )
    model_df = model_df.loc[model_df["narrative_present"].eq(1)].copy()
    return model_df, topic_cluster_features


def score_predictions(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def build_rf_pipeline(max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features="sqrt", selected_features=None):
    categorical_features = ["Product", "Sub-product", "Issue", "Sub-issue", "Company", "Submitted via"] + topic_cluster_features
    numeric_features = [
        "received_year",
        "received_month",
        "received_quarter",
        "received_dayofweek",
        "received_day",
        "narrative_present",
        "narrative_char_count",
        "narrative_word_count",
        "narrative_sentiment_score",
    ]
    if selected_features is not None:
        selected_set = set(selected_features)
        categorical_features = [f for f in categorical_features if f in selected_set]
        numeric_features = [f for f in numeric_features if f in selected_set]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                        ("onehot", make_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
            (
                "num",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric_features,
            ),
        ]
    )

    model = RandomForestClassifier(
        n_estimators=225,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    return pipeline, categorical_features, numeric_features


In [ ]:
model_df, topic_cluster_features = prepare_model_df(raw_df)
target_col = "target_relief"
rf_pipeline, rf_categorical, rf_numeric = build_rf_pipeline()
rf_features = rf_categorical + rf_numeric

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE,
)
X_train = train_df[rf_features]
y_train = train_df[target_col]
X_test = test_df[rf_features]
y_test = test_df[target_col]

if COMPARISON_PATH.exists():
    comparison_df = pd.read_csv(COMPARISON_PATH)
    rf_row = comparison_df.loc[comparison_df["model_family"].eq("Random Forest")]
    if not rf_row.empty:
        best_params = ast.literal_eval(rf_row.iloc[0]["best_params"])
        best_rf_params = {
            "max_depth": best_params.get("model__max_depth"),
            "min_samples_leaf": best_params.get("model__min_samples_leaf", 1),
            "min_samples_split": best_params.get("model__min_samples_split", 2),
            "max_features": best_params.get("model__max_features", "sqrt"),
        }
    else:
        best_rf_params = {"max_depth": 22, "min_samples_leaf": 5, "min_samples_split": 5, "max_features": "sqrt"}
else:
    best_rf_params = {"max_depth": 22, "min_samples_leaf": 5, "min_samples_split": 5, "max_features": "sqrt"}

rf_pipeline, rf_categorical, rf_numeric = build_rf_pipeline(**best_rf_params)
rf_features = rf_categorical + rf_numeric
X_train = train_df[rf_features]
X_test = test_df[rf_features]

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    scoring=scoring,
    cv=cv,
    n_jobs=N_JOBS,
    return_train_score=False,
)

rf_pipeline.fit(X_train, y_train)
test_proba = rf_pipeline.predict_proba(X_test)[:, 1]
test_pred = rf_pipeline.predict(X_test)
holdout_metrics = score_predictions(y_test, test_pred, test_proba)

cv_summary = pd.DataFrame(
    {
        "metric": list(scoring.keys()),
        "cv_mean": [cv_results[f"test_{m}"].mean() for m in scoring],
        "cv_std": [cv_results[f"test_{m}"].std() for m in scoring],
        "holdout": [holdout_metrics[m] for m in scoring],
    }
).sort_values("metric").reset_index(drop=True)

cv_summary.round(4).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_random_forest_cv_summary.csv", index=False)
display(cv_summary)
print(f"Using features: {rf_features}")
print(f"Detected topic-cluster features: {topic_cluster_features if topic_cluster_features else 'none yet'}")
print(classification_report(y_test, test_pred, digits=4))


This section provides the core evaluation summary for the final narrative-subset Random Forest. The key comparison is not just absolute performance, but whether this narrative-aware workflow outperforms the all-records baseline when both are judged on complaints that include narratives.


### Feature Importance

Feature importance and ablation are especially useful here because the Random Forest can capture nonlinear interactions, but that flexibility also makes it harder to explain by inspection. We therefore look at both built-in tree importance and permutation-based importance.


In [ ]:
fitted_preprocessor = rf_pipeline.named_steps["preprocessor"]
fitted_model = rf_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": fitted_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance_df["feature_group"] = (
    feature_importance_df["feature"]
    .str.split("__").str[1]
    .str.replace(r"^(.*?)_.*$", r"\1", regex=True)
)
group_importance_df = (
    feature_importance_df.groupby("feature_group", dropna=False)["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

perm_subset_X = X_test.sample(n=min(4000, len(X_test)), random_state=RANDOM_STATE)
perm_subset_y = y_test.loc[perm_subset_X.index]
perm = permutation_importance(
    rf_pipeline,
    perm_subset_X,
    perm_subset_y,
    scoring="average_precision",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
perm_df = pd.DataFrame(
    {
        "feature": rf_features,
        "permutation_importance_mean": perm.importances_mean,
        "permutation_importance_std": perm.importances_std,
    }
).sort_values("permutation_importance_mean", ascending=False)

feature_importance_df.head(20).round(6).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_feature_importance_top20.csv", index=False)
group_importance_df.round(6).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_feature_group_importance.csv", index=False)
perm_df.round(6).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_permutation_importance.csv", index=False)
display(feature_importance_df.head(20))
display(group_importance_df)
display(perm_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=feature_importance_df.head(15), x="importance", y="feature", ax=axes[0], color="#4C78A8")
axes[0].set_title("Top 15 Tree-Based Feature Importances")

sns.barplot(data=perm_df.head(10), x="permutation_importance_mean", y="feature", ax=axes[1], color="#F58518")
axes[1].set_title("Permutation Importance on Holdout Sample (Average Precision)")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "04_final_narrative_feature_importance_plots.png", dpi=200, bbox_inches="tight")


A strong evaluation should connect importance results back to modeling choices. In the final narrative-only notebook, the key question is whether complaint context, company context, and lightweight text features explain most of the monetary-relief signal once we limit the task to records where narratives are available for everyone in scope.


### Ablation Analysis

Ablation testing measures how much the model depends on different groups of features. To keep runtime reasonable, this section uses a 3-fold CV summary on the training set and compares F1 after removing one feature family at a time.


In [ ]:
feature_groups = {
    "all_features": rf_features,
    "no_narrative_size": [f for f in rf_features if f not in {"narrative_char_count", "narrative_word_count"}],
    "no_sentiment": [f for f in rf_features if f != "narrative_sentiment_score"],
    "no_company": [f for f in rf_features if f != "Company"],
    "no_product_context": [f for f in rf_features if f not in {"Product", "Sub-product"}],
    "no_issue_context": [f for f in rf_features if f not in {"Issue", "Sub-issue"}],
}
if topic_cluster_features:
    feature_groups["no_topic_cluster"] = [f for f in rf_features if f not in set(topic_cluster_features)]

ablation_rows = []
fast_cv = StratifiedKFold(n_splits=FAST_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for label, cols in feature_groups.items():
    ablation_pipeline, _, _ = build_rf_pipeline(selected_features=cols, **best_rf_params)
    scores = cross_validate(
        ablation_pipeline,
        train_df[cols],
        y_train,
        scoring={"f1": "f1", "recall": "recall", "precision": "precision"},
        cv=fast_cv,
        n_jobs=N_JOBS,
        return_train_score=False,
    )
    ablation_rows.append(
        {
            "ablation": label,
            "cv_mean_f1": scores["test_f1"].mean(),
            "cv_mean_precision": scores["test_precision"].mean(),
            "cv_mean_recall": scores["test_recall"].mean(),
            "feature_count": len(cols),
        }
    )

ablation_df = pd.DataFrame(ablation_rows).sort_values("cv_mean_f1", ascending=False).reset_index(drop=True)
baseline_f1 = ablation_df.loc[ablation_df["ablation"].eq("all_features"), "cv_mean_f1"].iloc[0]
ablation_df["f1_drop_vs_all_features"] = baseline_f1 - ablation_df["cv_mean_f1"]
ablation_df.round(4).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_ablation_results.csv", index=False)
display(ablation_df)


### Sensitivity Analysis

Sensitivity analysis helps us understand whether the selected Random Forest is stable or fragile. Here we vary two influential hyperparameters, `max_depth` and `min_samples_leaf`, and track cross-validated F1. A stable model should not collapse from small changes around the selected setting.


In [ ]:
sensitivity_rows = []
sensitivity_train_df = train_df.sample(n=min(25000, len(train_df)), random_state=RANDOM_STATE)
sensitivity_y = sensitivity_train_df[target_col]
for max_depth in [14, 22, None]:
    for min_samples_leaf in [2, 5, 10]:
        pipeline, _, _ = build_rf_pipeline(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=best_rf_params["min_samples_split"],
            max_features=best_rf_params["max_features"],
        )
        scores = cross_validate(
            pipeline,
            sensitivity_train_df[rf_features],
            sensitivity_y,
            scoring={"f1": "f1", "recall": "recall", "precision": "precision"},
            cv=fast_cv,
            n_jobs=N_JOBS,
            return_train_score=False,
        )
        sensitivity_rows.append(
            {
                "max_depth": str(max_depth),
                "min_samples_leaf": min_samples_leaf,
                "cv_mean_f1": scores["test_f1"].mean(),
                "cv_mean_precision": scores["test_precision"].mean(),
                "cv_mean_recall": scores["test_recall"].mean(),
            }
        )

sensitivity_df = pd.DataFrame(sensitivity_rows).sort_values(["cv_mean_f1", "cv_mean_recall"], ascending=False)
sensitivity_df.round(4).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_sensitivity_results.csv", index=False)
display(sensitivity_df)

pivot_f1 = sensitivity_df.pivot(index="min_samples_leaf", columns="max_depth", values="cv_mean_f1")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="Blues")
plt.title("Random Forest Sensitivity: CV F1")
plt.ylabel("min_samples_leaf")
plt.xlabel("max_depth")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "04_final_narrative_sensitivity_heatmap.png", dpi=200, bbox_inches="tight")


### Tradeoff Analysis

Tradeoff analysis asks how the Random Forest behaves when we change decision thresholds or training data size. This is still useful even with a more balanced target, because threshold shifts can change the precision-recall tradeoff materially.


In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, test_proba)
threshold_f1 = np.divide(
    2 * precisions[:-1] * recalls[:-1],
    precisions[:-1] + recalls[:-1],
    out=np.zeros_like(precisions[:-1]),
    where=(precisions[:-1] + recalls[:-1]) != 0,
)
best_threshold_idx = int(np.argmax(threshold_f1))
best_threshold = float(thresholds[best_threshold_idx])
threshold_pred = (test_proba >= best_threshold).astype(int)
threshold_metrics = score_predictions(y_test, threshold_pred, test_proba)

threshold_tradeoff_df = pd.DataFrame(
    {
        "setting": ["default_0.50", f"best_f1_{best_threshold:.3f}"],
        "accuracy": [holdout_metrics["accuracy"], threshold_metrics["accuracy"]],
        "precision": [holdout_metrics["precision"], threshold_metrics["precision"]],
        "recall": [holdout_metrics["recall"], threshold_metrics["recall"]],
        "f1": [holdout_metrics["f1"], threshold_metrics["f1"]],
        "balanced_accuracy": [holdout_metrics["balanced_accuracy"], threshold_metrics["balanced_accuracy"]],
        "average_precision": [holdout_metrics["average_precision"], threshold_metrics["average_precision"]],
    }
)
threshold_tradeoff_df.round(4).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_threshold_tradeoff.csv", index=False)
display(threshold_tradeoff_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix at 0.50 Threshold")
ConfusionMatrixDisplay.from_predictions(y_test, threshold_pred, ax=axes[1], colorbar=False)
axes[1].set_title(f"Confusion Matrix at Best F1 Threshold ({best_threshold:.3f})")
plt.tight_layout()
fig.savefig(VISUALS_DIR / "04_final_narrative_confusion_matrices.png", dpi=200, bbox_inches="tight")

plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions, label="Precision-Recall Curve")
plt.scatter(recalls[best_threshold_idx], precisions[best_threshold_idx], color="red", label=f"Best F1 threshold = {best_threshold:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve for Random Forest")
plt.legend()
plt.tight_layout()
plt.savefig(VISUALS_DIR / "04_final_narrative_precision_recall_curve.png", dpi=200, bbox_inches="tight")


In [ ]:
curve_train_df = train_df.sample(n=min(25000, len(train_df)), random_state=RANDOM_STATE)
curve_y = curve_train_df[target_col]
curve_pipeline, _, _ = build_rf_pipeline(**best_rf_params)
train_sizes, train_scores, valid_scores = learning_curve(
    curve_pipeline,
    curve_train_df[rf_features],
    curve_y,
    train_sizes=np.linspace(0.2, 1.0, 5),
    cv=fast_cv,
    scoring="f1",
    n_jobs=N_JOBS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

learning_curve_df = pd.DataFrame(
    {
        "train_size": train_sizes,
        "train_f1_mean": train_scores.mean(axis=1),
        "validation_f1_mean": valid_scores.mean(axis=1),
    }
)
learning_curve_df.round(4).to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_learning_curve.csv", index=False)
display(learning_curve_df)

plt.figure(figsize=(7, 5))
plt.plot(learning_curve_df["train_size"], learning_curve_df["train_f1_mean"], marker="o", label="Train F1")
plt.plot(learning_curve_df["train_size"], learning_curve_df["validation_f1_mean"], marker="o", label="Validation F1")
plt.xlabel("Training Examples")
plt.ylabel("F1 Score")
plt.title("Training Data Curve for Random Forest")
plt.legend()
plt.tight_layout()
plt.savefig(VISUALS_DIR / "04_final_narrative_learning_curve.png", dpi=200, bbox_inches="tight")


This section captures multiple project-required tradeoffs:

- precision versus recall as the classification threshold changes
- conservative default thresholding versus threshold tuning for minority-class detection
- training data size versus validation F1 through the learning curve

Together, these analyses help explain whether the model is limited more by representation, thresholding, or simply available data.


### Failure Analysis

Failure analysis focuses on specific mistakes rather than aggregate metrics. To satisfy the assignment requirement, we select at least three concrete failed predictions and group them into different categories. The goal is not just to note the error, but to reason about why it likely occurred and what future improvements could address it.


In [ ]:
failure_df = test_df[[
    "Complaint ID",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company",
    "Submitted via",
    "Consumer complaint narrative",
    target_col,
]].copy()
failure_df["predicted_label"] = test_pred
failure_df["predicted_proba"] = test_proba
failure_df["error_type"] = np.where(
    (failure_df[target_col] == 1) & (failure_df["predicted_label"] == 0),
    "false_negative",
    np.where(
        (failure_df[target_col] == 0) & (failure_df["predicted_label"] == 1),
        "false_positive",
        "correct",
    ),
)
failure_df["narrative_present"] = failure_df["Consumer complaint narrative"].fillna("").str.len().gt(0)
failure_df["narrative_word_count"] = failure_df["Consumer complaint narrative"].fillna("").str.split().str.len().fillna(0)

false_negative_examples = failure_df.loc[failure_df["error_type"].eq("false_negative")].sort_values("predicted_proba", ascending=False)
false_positive_examples = failure_df.loc[failure_df["error_type"].eq("false_positive")].sort_values("predicted_proba", ascending=False)

failure_examples = pd.concat(
    [
        false_negative_examples.head(1).assign(failure_category="likely systematic minority-class miss"),
        false_positive_examples.head(1).assign(failure_category="aggressive positive flag / ambiguous case"),
        false_negative_examples.sort_values("narrative_word_count").head(1).assign(failure_category="sparse-information edge case"),
    ],
    axis=0,
).drop_duplicates(subset=["Complaint ID"]).reset_index(drop=True)

failure_examples.to_csv(OUTPUT_TABLES_DIR / "04_final_narrative_failure_examples.csv", index=False)
display(failure_examples[[
    "Complaint ID",
    "failure_category",
    "error_type",
    target_col,
    "predicted_label",
    "predicted_proba",
    "Product",
    "Issue",
    "Sub-issue",
    "Company",
    "Submitted via",
    "narrative_present",
    "narrative_word_count",
]])


Use the displayed failure cases to write the final report narrative in a structured way:

1. Describe the record and the incorrect prediction.
2. Note whether the failure looks more like a systematic minority-class miss, an edge case with weak information, or an overly aggressive positive flag.
3. Connect that failure back to evidence from feature importance, threshold analysis, or data imbalance.
4. Propose a future fix, such as threshold adjustment, richer narrative features, additional data, or different sampling/ensemble strategies.

Practical interpretation hints:

- False negatives usually support the argument that the default threshold is conservative or that rare-class signal is underrepresented.
- False positives often suggest overlapping patterns between timely and untimely complaints for certain issue or company contexts.
- Very short or missing narratives can create edge cases because the model has less descriptive context available, even when simple narrative indicators are included.


### Conclusions

This notebook covers the main evaluation requirements for the selected final narrative-only Random Forest.

The most important interpretation is comparative:

- does the narrative-only model beat the all-records baseline on the narrative-present subset?
- does sentiment help enough to justify keeping it?
- do future topic-cluster features look worth integrating into this narrative-only path?

If the answer is yes, then a two-track strategy becomes easy to justify: a universal structured model for all complaints, plus a narrative-aware model for complaints that include narrative text.
